In [ ]:
!apt-get install graphviz -y
!pip install graphviz pandas openpyxl

1. The Solver Module (solver.py)
Description: This module acts as the "Brain." It contains the pure mathematical logic required to solve the puzzle. It maintains the state of the grid, tracks metrics (backtracks, nodes visited), and records the decision path (graph edges) for later visualization. Crucially, this module does not print to the console or save files; it only processes data. It implements recursive backtracking with optional heuristics (MVC) and optimization (Lookahead).


```
# This is formatted as code
CLASS SudokuSolver:

    FUNCTION Init(raw_board_data):
        Initialize empty 9x9 grid
        Initialize metrics (backtracks = 0, graph_edges = [])
        Parse raw_board_data into integers on the grid
        Mark non-zero numbers as "Original Clues" (for color tracking later)

    FUNCTION is_safe(row, col, number):
        IF number exists in current Row OR Column OR 3x3 Box:
            RETURN False
        ELSE:
            RETURN True

    FUNCTION find_best_empty_cell(use_mvc_flag):
        IF use_mvc_flag IS False:
            RETURN first empty cell found (Linear Search)
        
        ELSE (MVC Heuristic):
            Scan all empty cells
            Calculate "Degree of Freedom" (how many valid numbers fit there)
            RETURN the cell with the FEWEST valid options

    FUNCTION check_lookahead_fail(row, col):
        (Hypothetically check neighbors after placing a number)
        FOR every neighbor of (row, col):
            IF neighbor is empty AND has NO valid candidates remaining:
                RETURN True (Failure detected)
        RETURN False

    FUNCTION solve(use_mvc, use_lookahead):
        INCREMENT node_visited_count

        Target_Cell = CALL find_best_empty_cell(use_mvc)
        
        IF no Target_Cell exists:
            RETURN True (Puzzle Solved)

        FOR Number from 1 to 9:
            IF is_safe(Target_Cell, Number):
                
                RECORD edge in graph_edges (Parent_Node -> Current_Node)
                PLACE Number in Grid

                IF use_lookahead IS True:
                    IF check_lookahead_fail(Target_Cell) IS True:
                        RESET Cell to 0
                        CONTINUE to next number (Skip recursion)

                IF solve(use_mvc, use_lookahead) returns True:
                    RETURN True (Bubble up success)

                (Backtracking Step)
                RESET Cell to 0
                INCREMENT backtrack_count
                RECORD "Backtrack" edge in graph_edges

        RETURN False (Trigger backtrack in parent)
```


In [ ]:
%%writefile solver.py
import sys
import json
import os
import random
import copy

# Default path for the config file
CONFIG_PATH = "config.json"

class SudokuSolver:
    def __init__(self, initial_board=None, config=None):
        # Allow passing config directly to avoid re-loading from file every time
        self.config = config if config else self.load_config()

        # Increase recursion depth for deep brute force trees
        if "METRICS_CONSTANTS" in self.config and "RECURSION_LIMIT" in self.config["METRICS_CONSTANTS"]:
            sys.setrecursionlimit(self.config["METRICS_CONSTANTS"]["RECURSION_LIMIT"])

        # Grid Constants
        N = self.config["GRID_CONSTANTS"]["SIZE"]
        DOMAIN_MIN = self.config["GRID_CONSTANTS"]["DOMAIN_MIN"]
        DOMAIN_MAX = self.config["GRID_CONSTANTS"]["DOMAIN_MAX"]

        self.domain = set(range(DOMAIN_MIN, DOMAIN_MAX + 1))
        self.grid = [[0 for _ in range(N)] for _ in range(N)]
        self.is_original = [[False for _ in range(N)] for _ in range(N)]

        self.metrics = {
            "backtracks": 0,
            "nodes_visited": 0,
            "graph_edges": [],
            "node_data": {}, # Stores metadata (candidates, coordinates) for each node
            "node_counter": 0
        }

        if initial_board:
            self.parse_board(initial_board)

    def load_config(self):
        """Loads configuration from config.json."""
        try:
            with open(CONFIG_PATH, 'r') as f:
                return json.load(f)
        except FileNotFoundError:
            print(f"Error: {CONFIG_PATH} not found.")
            return {
                "GRID_CONSTANTS": {"SIZE": 9, "BLOCK_SIZE": 3, "DOMAIN_MIN": 1, "DOMAIN_MAX": 9, "EMPTY_CELL_CHARS": ["0"]},
                "METRICS_CONSTANTS": {"GRAPH_LIMIT": 60, "RECURSION_LIMIT": 5000}
            }

    def parse_board(self, raw_data):
        N = self.config["GRID_CONSTANTS"]["SIZE"]
        CLUE_CHARS = set(self.config["GRID_CONSTANTS"]["CLUE_CHARS"])
        EMPTY_CELL_CHARS = set(self.config["GRID_CONSTANTS"]["EMPTY_CELL_CHARS"])

        flat_digits = []

        # Handle string or list input
        if isinstance(raw_data, list):
            if raw_data and isinstance(raw_data[0], list):
                flat_digits = [item for sublist in raw_data for item in sublist]
            else:
                flat_digits = raw_data
        else:
            raw_string = str(raw_data)
            for char in raw_string:
                if char.isdigit() and char in CLUE_CHARS:
                    flat_digits.append(int(char))
                elif char in EMPTY_CELL_CHARS:
                    flat_digits.append(0)

        while len(flat_digits) < N * N: flat_digits.append(0)

        self.grid = [flat_digits[i:i+N] for i in range(0, N * N, N)]

        for r in range(N):
            for c in range(N):
                if self.grid[r][c] != 0:
                    self.is_original[r][c] = True

    def get_board_string(self):
        """Returns the current grid as a single string of digits."""
        flat = []
        for row in self.grid:
            flat.extend(map(str, row))
        return "".join(flat)

    def is_safe(self, row, col, num):
        N = self.config["GRID_CONSTANTS"]["SIZE"]
        B = self.config["GRID_CONSTANTS"]["BLOCK_SIZE"]

        if num in self.grid[row]: return False

        for i in range(N):
            if self.grid[i][col] == num: return False

        sr, sc = B * (row // B), B * (col // B)
        for i in range(B):
            for j in range(B):
                if self.grid[sr + i][sc + j] == num: return False

        return True

    def get_candidates(self, row, col):
        N = self.config["GRID_CONSTANTS"]["SIZE"]
        B = self.config["GRID_CONSTANTS"]["BLOCK_SIZE"]

        candidates = set(self.domain)

        for c in range(N): candidates.discard(self.grid[row][c])
        for r in range(N): candidates.discard(self.grid[r][col])

        sr, sc = B * (row // B), B * (col // B)
        for r in range(sr, sr + B):
            for c in range(sc, sc + B):
                candidates.discard(self.grid[r][c])

        return candidates

    def get_candidate_counts(self):
        """Returns a grid showing the number of candidates for each empty cell."""
        N = self.config["GRID_CONSTANTS"]["SIZE"]
        counts_grid = [[0 for _ in range(N)] for _ in range(N)]

        for r in range(N):
            for c in range(N):
                if self.grid[r][c] == 0:
                    counts_grid[r][c] = len(self.get_candidates(r, c))
                else:
                    counts_grid[r][c] = -1 # Mark filled cells
        return counts_grid

    def find_empty_cell_mvc(self):
        N = self.config["GRID_CONSTANTS"]["SIZE"]
        best_cell = None
        min_candidates = N + 1

        for r in range(N):
            for c in range(N):
                if self.grid[r][c] == 0:
                    count = len(self.get_candidates(r, c))
                    if count < min_candidates:
                        min_candidates = count
                        best_cell = (r, c)
                        if min_candidates <= 1: return best_cell
        return best_cell

    def check_lookahead_fail(self, row, col):
        """
        Checks if placing a number at (row, col) has caused any
        neighboring empty cell to have NO valid candidates left.
        """
        N = self.config["GRID_CONSTANTS"]["SIZE"]
        B = self.config["GRID_CONSTANTS"]["BLOCK_SIZE"]

        # Check Row
        for c_idx in range(N):
            if self.grid[row][c_idx] == 0 and len(self.get_candidates(row, c_idx)) == 0: return True

        # Check Column
        for r_idx in range(N):
            if self.grid[r_idx][col] == 0 and len(self.get_candidates(r_idx, col)) == 0: return True

        # Check Box
        sr, sc = B * (row // B), B * (col // B)
        for r in range(sr, sr + B):
            for c in range(sc, sc + B):
                if self.grid[r][c] == 0 and len(self.get_candidates(r, c)) == 0: return True

        return False

    def solve(self, use_mvc=False, use_lookahead=False, parent_id=0):
        N = self.config["GRID_CONSTANTS"]["SIZE"]
        DOMAIN_MIN = self.config["GRID_CONSTANTS"]["DOMAIN_MIN"]
        DOMAIN_MAX = self.config["GRID_CONSTANTS"]["DOMAIN_MAX"]
        GRAPH_LIMIT = self.config["METRICS_CONSTANTS"]["GRAPH_LIMIT"]

        self.metrics["nodes_visited"] += 1

        if use_mvc:
            empty_spot = self.find_empty_cell_mvc()
        else:
            empty_spot = None
            for i in range(N):
                for j in range(N):
                    if self.grid[i][j] == 0:
                        empty_spot = (i, j)
                        break
                if empty_spot: break

        if not empty_spot: return True

        row, col = empty_spot

        # --- VISUALIZATION UPDATE: Capture Domain ---
        if parent_id <= GRAPH_LIMIT:
             current_candidates = sorted(list(self.get_candidates(row, col)))
             self.metrics["node_data"][parent_id] = {
                 "row": row,
                 "col": col,
                 "candidates": current_candidates
             }

        for num in range(DOMAIN_MIN, DOMAIN_MAX + 1):
            if self.is_safe(row, col, num):

                self.metrics["node_counter"] += 1
                current_id = self.metrics["node_counter"]

                if current_id <= GRAPH_LIMIT:
                    self.metrics['graph_edges'].append((parent_id, current_id, f"Set({row},{col})={num}"))

                self.grid[row][col] = num

                if use_lookahead and self.check_lookahead_fail(row, col):
                    self.grid[row][col] = 0
                    continue

                if self.solve(use_mvc, use_lookahead, parent_id=current_id):
                    return True

                self.grid[row][col] = 0
                self.metrics["backtracks"] += 1

                if current_id <= GRAPH_LIMIT:
                    self.metrics["node_counter"] += 1
                    back_id = self.metrics["node_counter"]
                    self.metrics['graph_edges'].append((current_id, back_id, "Backtrack"))

        return False

    @staticmethod
    def generate_puzzle(holes, config):
        """
        Generates a new Sudoku puzzle with a specified number of holes.
        """
        N = config["GRID_CONSTANTS"]["SIZE"]
        B = config["GRID_CONSTANTS"]["BLOCK_SIZE"]

        # 1. Create an empty solver
        solver = SudokuSolver(config=config)

        # 2. Fill diagonal blocks (independent, so safe to randomize)
        for i in range(0, N, B):
            digits = list(range(config["GRID_CONSTANTS"]["DOMAIN_MIN"], config["GRID_CONSTANTS"]["DOMAIN_MAX"] + 1))
            random.shuffle(digits)
            for r in range(B):
                for c in range(B):
                    solver.grid[i + r][i + c] = digits.pop()

        # 3. Solve to fill the rest
        solver.solve(use_mvc=True) # Use efficient solver to fill
        full_grid = copy.deepcopy(solver.grid)

        # 4. Remove Digits
        attempts = holes
        while attempts > 0:
            r = random.randint(0, N-1)
            c = random.randint(0, N-1)
            if full_grid[r][c] != 0:
                full_grid[r][c] = 0
                attempts -= 1

        # Return puzzle string and solution string
        solver_final = SudokuSolver(config=config)
        solver_final.grid = full_grid
        puzzle_str = solver_final.get_board_string()

        solver_sol = SudokuSolver(config=config)
        solver_sol.grid = solver.grid # The full solved grid
        solution_str = solver_sol.get_board_string()

        return puzzle_str, solution_str

2. The Visualizer Module (visualizer.py)
Description: This module acts as the "View" and "IO Manager." It handles everything the user sees and the files the computer saves. It is responsible for:

   * Printing the board to the console with color coding (Green for valid new entries, White for original clues).

   * Rendering the decision tree using Graphviz.

   * Saving the batch results to TXT, CSV, or XLSX files.


```
# This is formatted as code

CLASS OutputManager:

    FUNCTION display_board(solver_instance, title, color_flag):
        PRINT title
        FOR each cell in grid:
            IF cell value is 0:
                PRINT "."
            ELSE IF cell is an "Original Clue":
                PRINT value in White
            ELSE (It is a solved number):
                IF color_flag IS True:
                    Check if value causes conflict
                    IF Valid -> PRINT value in Green
                    ELSE -> PRINT value in Red
                ELSE:
                    PRINT value in White

    FUNCTION render_graph(metrics):
        Initialize Graphviz Digraph
        Add Root Node
        
        FOR each edge in metrics['graph_edges']:
            (Limit to first 60 edges to prevent crashing viewer)
            IF edge is a "Backtrack":
                Draw Red Dashed Line
            ELSE:
                Draw Solid Line
        
        DISPLAY Graph Image

    FUNCTION save_batch(config, data_list, filename):
        Determine File Format from Config (TXT, CSV, or XLSX)
        Create Directory if it doesn't exist
        
        IF Format is TXT:
            WRITE header (Timestamp)
            FOR each result in data_list:
                WRITE Stats (Time, Backtracks, Mode)
                WRITE Ascii Grid representation
        
        ELSE IF Format is CSV or XLSX:
            Convert data_list to Data Table (Pandas DataFrame)
            Export to selected format
```

In [ ]:
%%writefile visualizer.py
import os
import pandas as pd
import numpy as np
import graphviz
import datetime
import json
from IPython.display import display, HTML
from typing import Dict

def load_config(path="config.json") -> Dict:
    try:
        with open(path, 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        return {}

class OutputManager:
    @staticmethod
    def ensure_directory(path):
        if not os.path.exists(path):
            os.makedirs(path)

    @staticmethod
    def display_board(solver, config, title="Sudoku Board", use_color=True):
        N = config["GRID_CONSTANTS"]["SIZE"]
        B = config["GRID_CONSTANTS"]["BLOCK_SIZE"]
        C_GREEN = config["COLORS"]["GREEN"]
        C_RED = config["COLORS"]["RED"]
        C_RESET = config["COLORS"]["RESET"]
        EMPTY_CHAR = config["COLORS"]["EMPTY_CHAR"]

        print(config["VISUAL_CONSTANTS"]["TITLE_SEPARATOR"].format(title=title))
        print(config["VISUAL_CONSTANTS"]["BOARD_TOP"])

        for i in range(N):
            print(config["VISUAL_CONSTANTS"]["CELL_EDGE"], end="")
            for j in range(N):
                val = solver.grid[i][j]

                if val == 0:
                    char_str = EMPTY_CHAR
                else:
                    if not use_color:
                        char_str = str(val)
                    elif solver.is_original[i][j]:
                        char_str = str(val)
                    else:
                        solver.grid[i][j] = 0
                        is_safe = solver.is_safe(i, j, val)
                        solver.grid[i][j] = val

                        if is_safe: char_str = f"{C_GREEN}{val}{C_RESET}"
                        else: char_str = f"{C_RED}{val}{C_RESET}"

                if (j + 1) % B == 0 and j < N - 1:
                    print(f"{char_str}{config['VISUAL_CONSTANTS']['BLOCK_SEPARATOR']}", end="")
                elif j == N - 1:
                    print(f"{char_str}{config['VISUAL_CONSTANTS']['ROW_END']}")
                else:
                    print(f"{char_str} ", end="")

            if (i + 1) % B == 0 and i < N - 1:
                print(config["VISUAL_CONSTANTS"]["BOARD_SEPARATOR"])

        print(config["VISUAL_CONSTANTS"]["BOARD_BOTTOM"])

    @staticmethod
    def render_mvc_heatmap(counts_grid, config):
        """
        Renders a heatmap showing the number of candidates for each cell.
        Useful for visualizing the Minimum Remaining Values heuristic.
        """
        print(config["VISUAL_CONSTANTS"]["TITLE_SEPARATOR"].format(title="MVC Candidate Counts"))

        N = config["GRID_CONSTANTS"]["SIZE"]
        df = pd.DataFrame(counts_grid)

        # Replace -1 (filled cells) with NaN for better styling
        df_display = df.replace(-1, np.nan)

        # Apply gradient: Red (low counts, urgent) to Green (high counts)
        # 'RdYlGn_r' reverse map means Low=Red, High=Green
        styled = df_display.style.background_gradient(cmap='RdYlGn_r', vmin=1, vmax=N)\
                             .format("{:.0f}", na_rep="")\
                             .set_caption("Minimum Remaining Values (MVC) Map (Red = Fewer Candidates)")\
                             .set_properties(**{
                                 'text-align': 'center',
                                 'border': '1px solid #555',
                                 'width': '30px',
                                 'height': '30px',
                                 'font-size': '14px'
                             })

        display(styled)

    @staticmethod
    def render_graph(metrics, config):
        """
        Uses Graphviz to render the decision tree.
        Updates: Shows candidates at each node and crosses out failed ones.
        """
        edges = metrics['graph_edges']
        node_data = metrics.get('node_data', {})
        GRAPH_LIMIT = config["METRICS_CONSTANTS"]["GRAPH_LIMIT"]
        V_CONST = config["VISUAL_CONSTANTS"]

        if not edges:
            print(V_CONST["LOG_NO_GRAPH"])
            return

        dot = graphviz.Digraph(comment=V_CONST["GRAPH_TITLE"])
        dot.attr(rankdir='TB')
        dot.attr('node', shape='plain') # HTML-like labels require shape=plain

        # 1. Pre-process to identify "Failed" paths
        failed_nodes = set()
        parent_child_map = {} # parent_id -> list of (child_id, value_attempted)

        for parent, child, label in edges:
            if "Backtrack" in label:
                failed_nodes.add(parent)
            elif "Set" in label:
                try:
                    val = int(label.split('=')[1])
                    if parent not in parent_child_map:
                        parent_child_map[parent] = []
                    parent_child_map[parent].append((child, val))
                except IndexError:
                    pass

        # 2. Draw Nodes
        all_nodes = set()
        for p, c, l in edges:
            all_nodes.add(p)
            all_nodes.add(c)

        for node_id in all_nodes:
            # Special Handling for nodes with captured data (Start and Decisions)
            if node_id in node_data:
                data = node_data[node_id]
                cands = data['candidates']
                children = parent_child_map.get(node_id, [])

                cand_html = []
                for c_val in cands:
                    # Check if this value led to a failure
                    is_failed = False
                    for (child, val) in children:
                        if val == c_val and child in failed_nodes:
                            is_failed = True
                            break

                    if is_failed:
                        cand_html.append(f'<S>{c_val}</S>') # Strikethrough
                    else:
                        cand_html.append(str(c_val))

                cand_str = ", ".join(cand_html)

                bg_color = V_CONST['GRAPH_START_NODE_COLOR'] if node_id == 0 else V_CONST['GRAPH_NORMAL_NODE_COLOR']
                node_title = "Start" if node_id == 0 else f"Node {node_id}"

                label = f'''<
                <TABLE BORDER="0" CELLBORDER="1" CELLSPACING="0" BGCOLOR="{bg_color}">
                <TR><TD><B>{node_title}</B></TD></TR>
                <TR><TD>Cell ({data['row']},{data['col']})</TD></TR>
                <TR><TD>Domain: {{ {cand_str} }}</TD></TR>
                </TABLE>
                >'''
                dot.node(str(node_id), label=label)

            # Nodes without data (Backtrack targets or generic)
            elif node_id == 0:
                 dot.node(str(node_id), "Start", shape=V_CONST["GRAPH_START_NODE_SHAPE"],
                             style='filled', fillcolor=V_CONST["GRAPH_START_NODE_COLOR"])

        # 3. Draw Edges
        count = 0
        for parent, child, label in edges:
            if count > GRAPH_LIMIT: break

            if V_CONST["GRAPH_BACKTRACK_LABEL"] in label:
                dot.edge(str(parent), str(child), label=label,
                         color=V_CONST["GRAPH_BACKTRACK_COLOR"],
                         style=V_CONST["GRAPH_BACKTRACK_STYLE"])
                dot.node(str(child), label="Back",
                         style='filled', fillcolor=V_CONST["GRAPH_BACKTRACK_NODE_COLOR"],
                         shape=V_CONST["GRAPH_BACKTRACK_NODE_SHAPE"])
            else:
                dot.edge(str(parent), str(child), label=label)
            count += 1

        print(V_CONST["LOG_GRAPH_RENDERED"].format(count=count))
        display(dot)

    @staticmethod
    def generate_filename(config, p_name, mode, timestamp, iteration_idx=None):
        settings = config['output_settings']['filename_config']
        parts = ["sudokusolver_unittest"]

        if settings['include_testset_name']:
            parts.append(f"testset{p_name.replace(' ', '')}")

        if settings['include_iteration_index'] and iteration_idx is not None:
            parts.append(f"iteration{iteration_idx}")
        elif not config['output_settings']['batch_iterations'] and iteration_idx is not None:
            parts.append(f"iteration{iteration_idx}")

        if settings['include_modifiers']:
            flag_m = "1" if mode['mvc'] else "0"
            flag_l = "1" if mode['look'] else "0"
            parts.append(f"modifiers{flag_m}{flag_l}")

        if settings['include_date']:
            parts.append(f"date{timestamp}")

        filename = "_".join(parts)
        ext = config['output_settings']['file_format'].lower()
        return f"{filename}.{ext}"

    @staticmethod
    def save_batch(config, batch_data, filename):
        output_dir = config['output_settings']['directory']
        OutputManager.ensure_directory(output_dir)
        filepath = os.path.join(output_dir, filename)
        fmt = config['output_settings']['file_format'].lower()
        V_CONST = config["VISUAL_CONSTANTS"]

        if fmt == 'txt':
            with open(filepath, "w") as f:
                f.write(f"{V_CONST['REPORT_HEADER']}\n")
                f.write(f"{V_CONST['REPORT_TIMESTAMP_KEY']} {datetime.datetime.now()}\n")
                for entry in batch_data:
                    f.write("\n" + "-"*40 + "\n")
                    f.write(f"{V_CONST['REPORT_PUZZLE_KEY']} {entry['puzzle']}\n")
                    mode_str = V_CONST['REPORT_MODE_KEY'].format(
                        mode_name=entry['mode_name'],
                        mvc=entry['mvc'],
                        look=entry['look']
                    )
                    f.write(f"{mode_str}\n")
                    f.write(f"{V_CONST['REPORT_ITERATION_KEY']} {entry['iteration']}\n")
                    time_str = V_CONST['REPORT_TIME_KEY'].format(time=entry['time'])
                    f.write(f"{time_str}\n")
                    f.write(f"{V_CONST['REPORT_BACKTRACK_KEY']} {entry['backtracks']}\n")
                    f.write(f"{V_CONST['REPORT_GRID_KEY']}\n{entry['grid_str']}\n")
            print(V_CONST["LOG_SAVED_TXT"].format(filepath=filepath))

        else:
            df = pd.DataFrame(batch_data)
            if fmt == 'csv':
                df.to_csv(filepath, index=False)
                print(V_CONST["LOG_SAVED_CSV"].format(filepath=filepath))
            elif fmt == 'xlsx':
                df.to_excel(filepath, index=False)
                print(V_CONST["LOG_SAVED_XLSX"].format(filepath=filepath))
            else:
                print(V_CONST["LOG_UNKNOWN_FORMAT"].format(fmt=fmt))

Here is the comprehensive documentation and human-readable pseudocode for your modular Sudoku project. This breakdown explains the logic behind each module without getting bogged down in Python syntax.

1. The Solver Module (solver.py)
Description: This module acts as the "Brain." It contains the pure mathematical logic required to solve the puzzle. It maintains the state of the grid, tracks metrics (backtracks, nodes visited), and records the decision path (graph edges) for later visualization. Crucially, this module does not print to the console or save files; it only processes data. It implements recursive backtracking with optional heuristics (MVC) and optimization (Lookahead).

Shutterstock

Pseudocode:

Plaintext

CLASS SudokuSolver:

    FUNCTION Init(raw_board_data):
        Initialize empty 9x9 grid
        Initialize metrics (backtracks = 0, graph_edges = [])
        Parse raw_board_data into integers on the grid
        Mark non-zero numbers as "Original Clues" (for color tracking later)

    FUNCTION is_safe(row, col, number):
        IF number exists in current Row OR Column OR 3x3 Box:
            RETURN False
        ELSE:
            RETURN True

    FUNCTION find_best_empty_cell(use_mvc_flag):
        IF use_mvc_flag IS False:
            RETURN first empty cell found (Linear Search)
        
        ELSE (MVC Heuristic):
            Scan all empty cells
            Calculate "Degree of Freedom" (how many valid numbers fit there)
            RETURN the cell with the FEWEST valid options

    FUNCTION check_lookahead_fail(row, col):
        (Hypothetically check neighbors after placing a number)
        FOR every neighbor of (row, col):
            IF neighbor is empty AND has NO valid candidates remaining:
                RETURN True (Failure detected)
        RETURN False

    FUNCTION solve(use_mvc, use_lookahead):
        INCREMENT node_visited_count

        Target_Cell = CALL find_best_empty_cell(use_mvc)
        
        IF no Target_Cell exists:
            RETURN True (Puzzle Solved)

        FOR Number from 1 to 9:
            IF is_safe(Target_Cell, Number):
                
                RECORD edge in graph_edges (Parent_Node -> Current_Node)
                PLACE Number in Grid

                IF use_lookahead IS True:
                    IF check_lookahead_fail(Target_Cell) IS True:
                        RESET Cell to 0
                        CONTINUE to next number (Skip recursion)

                IF solve(use_mvc, use_lookahead) returns True:
                    RETURN True (Bubble up success)

                (Backtracking Step)
                RESET Cell to 0
                INCREMENT backtrack_count
                RECORD "Backtrack" edge in graph_edges

        RETURN False (Trigger backtrack in parent)
2. The Visualizer Module (visualizer.py)
Description: This module acts as the "View" and "IO Manager." It handles everything the user sees and the files the computer saves. It is responsible for:

Printing the board to the console with color coding (Green for valid new entries, White for original clues).

Rendering the decision tree using Graphviz.

Saving the batch results to TXT, CSV, or XLSX files.

Pseudocode:

Plaintext

CLASS OutputManager:

    FUNCTION display_board(solver_instance, title, color_flag):
        PRINT title
        FOR each cell in grid:
            IF cell value is 0:
                PRINT "."
            ELSE IF cell is an "Original Clue":
                PRINT value in White
            ELSE (It is a solved number):
                IF color_flag IS True:
                    Check if value causes conflict
                    IF Valid -> PRINT value in Green
                    ELSE -> PRINT value in Red
                ELSE:
                    PRINT value in White

    FUNCTION render_graph(metrics):
        Initialize Graphviz Digraph
        Add Root Node
        
        FOR each edge in metrics['graph_edges']:
            (Limit to first 60 edges to prevent crashing viewer)
            IF edge is a "Backtrack":
                Draw Red Dashed Line
            ELSE:
                Draw Solid Line
        
        DISPLAY Graph Image

    FUNCTION save_batch(config, data_list, filename):
        Determine File Format from Config (TXT, CSV, or XLSX)
        Create Directory if it doesn't exist
        
        IF Format is TXT:
            WRITE header (Timestamp)
            FOR each result in data_list:
                WRITE Stats (Time, Backtracks, Mode)
                WRITE Ascii Grid representation
        
        ELSE IF Format is CSV or XLSX:
            Convert data_list to Data Table (Pandas DataFrame)
            Export to selected format
3. The Generator Module (generator.py)
Description: This module creates the test data. It generates a valid, randomized Sudoku puzzle by first creating a complete solution and then removing a specific number of digits ("holes") to create a puzzle of varying difficulty.


```
# This is formatted as code
```

FUNCTION generate_test_puzzle(number_of_holes):
    
    Initialize a new SudokuSolver
    
    (Step 1: Randomize)
    Fill the three diagonal 3x3 boxes with random numbers 1-9
    (Diagonal boxes are independent, so this is safe and fast)

    (Step 2: Solve)
    CALL solver.solve(use_mvc=True) to fill the rest of the board
    
    (Step 3: Poke Holes)
    Create a copy of the full grid
    WHILE number_of_holes > 0:
        Pick random coordinates (row, col)
        IF cell is not empty:
            Set cell to 0
            DECREMENT number_of_holes

    RETURN flattened string of the puzzle

In [ ]:
%%writefile generator.py
import random
from solver import SudokuSolver

def generate_test_puzzle(difficulty_holes, config):
    gen_solver = SudokuSolver()
    gen_solver.solve()

    # Determine separator based on top-level config flag
    sep = "," if config.get('use_commas', False) else ""

    # 1. Create SOLVED string
    flat_solved = [str(val) for row in gen_solver.grid for val in row]
    solved_str = sep.join(flat_solved)

    # 2. Create UNSOLVED grid
    test_grid = [row[:] for row in gen_solver.grid]
    holes = difficulty_holes

    # --- CORRECT PATHING START ---
    # Access the nested keys correctly
    target_char = config['VISUAL_CONSTANTS']['GENERATOR_CHAR']
    empty_chars = config['GRID_CONSTANTS']['EMPTY_CELL_CHARS']
    # --- CORRECT PATHING END ---

    while holes > 0:
        r, c = random.randint(0, 8), random.randint(0, 8)

        # Check if the cell is NOT already one of the empty characters
        # Use the 'empty_chars' variable we defined above
        if str(test_grid[r][c]) not in empty_chars:
            test_grid[r][c] = target_char
            holes -= 1

    # 3. Create UNSOLVED string
    flat_unsolved = [str(val) for row in test_grid for val in row]
    unsolved_str = sep.join(flat_unsolved)

    return unsolved_str, solved_str

**The Configuration (config.json)**

**Description: **This is a static JSON file that controls the behavior of the entire suite. No code logic exists here, only settings.

* **`iterations:`** How many times to run each test (to get an average time).

*   **`visualize_graph:`** Boolean (True/False). If True, the decision tree is drawn in the notebook.

* **`use_color_output:`** Boolean. If True, console output uses Green/Red/White text.

* **output_settings:**

  * `directory:` Folder name for saved files.

  * `file_format: `Output type (txt, csv, xlsx).

  * `batch_iterations:` If True, combines all iterations of a test into one file. If False, saves 1 file per iteration.

  * `filename_config:` Flags to determine what text goes into the saved filename (e.g., date, mode, test name).

* `puzzles_to_generate: `A list of puzzles to create (e.g., "Easy" with 30 holes).

* `modes_to_test:` A list of algorithm configurations (e.g., "Base Brute Force", "MVC Only").





In [ ]:
%%writefile config.json
{
    "iterations": 3,
    "visualize_graph": true,
    "use_color_output": true,
    "use_commas": true,

    "GRID_CONSTANTS": {
        "SIZE": 9,
        "BLOCK_SIZE": 3,
        "DOMAIN_MIN": 1,
        "DOMAIN_MAX": 9,
        "EMPTY_CELL_CHARS": ["0", ".", "*", "-", "?"],
        "CLUE_CHARS": ["1", "2", "3", "4", "5", "6", "7", "8", "9"]
    },

    "METRICS_CONSTANTS": {
        "GRAPH_LIMIT": 60,
        "RECURSION_LIMIT": 5000
    },

    "COLORS": {
        "GREEN": "\u001b[92m",
        "RED": "\u001b[91m",
        "BLUE": "\u001b[94m",
        "YELLOW": "\u001b[93m",
        "RESET": "\u001b[0m",
        "BOLD": "\u001b[1m",
        "CLUE_COLOR": "",
        "EMPTY_CHAR": "."
    },

    "VISUAL_CONSTANTS": {
        "show_generated_puzzles": true,
        "GENERATOR_CHAR": "*",
        "TITLE_SEPARATOR": "\n--- {title} ---",
        "BOARD_TOP": "╔═══════╦═══════╦═══════╗",
        "BOARD_SEPARATOR": "╠═══════╬═══════╬═══════╣",
        "BOARD_BOTTOM": "╚═══════╩═══════╩═══════╝",
        "CELL_EDGE": "║ ",
        "BLOCK_SEPARATOR": " ║ ",
        "ROW_END": " ║ ",
        "GRAPH_TITLE": "Sudoku Decision Tree",
        "GRAPH_START_NODE": "Start",
        "GRAPH_BACKTRACK_LABEL": "Backtrack",
        "GRAPH_BACKTRACK_COLOR": "red",
        "GRAPH_BACKTRACK_STYLE": "dashed",
        "GRAPH_BACKTRACK_NODE_COLOR": "pink",
        "GRAPH_BACKTRACK_NODE_SHAPE": "box",
        "GRAPH_NORMAL_NODE_COLOR": "lightblue",
        "GRAPH_NORMAL_NODE_SHAPE": "ellipse",
        "GRAPH_START_NODE_COLOR": "orange",
        "GRAPH_START_NODE_SHAPE": "doublecircle",
        "LOG_NO_GRAPH": "[Visualizer] No graph edges to render.",
        "LOG_GRAPH_RENDERED": "[Visualizer] Graphing first {count} nodes:",
        "REPORT_HEADER": "=== UNIT TEST REPORT ===",
        "REPORT_TIMESTAMP_KEY": "Timestamp:",
        "REPORT_PUZZLE_KEY": "Puzzle:",
        "REPORT_MODE_KEY": "Mode: {mode_name} (MVC={mvc}, Look={look})",
        "REPORT_ITERATION_KEY": "Iteration:",
        "REPORT_TIME_KEY": "Time: {time:.5f}s",
        "REPORT_BACKTRACK_KEY": "Backtracks:",
        "REPORT_GRID_KEY": "Grid:",
        "LOG_SAVED_TXT": "   [ARTIFACT] Saved TXT: {filepath}",
        "LOG_SAVED_CSV": "   [ARTIFACT] Saved CSV: {filepath}",
        "LOG_SAVED_XLSX": "   [ARTIFACT] Saved XLSX: {filepath}",
        "LOG_UNKNOWN_FORMAT": "   [ERROR] Unknown format: {fmt}",

        "TEXT_TEMPLATES": {
            "MSG_GENERATING": "   >>> Generating Test Puzzles...",
            "SEP_EQUALS": "==================================================",
            "SEP_DASHES": "--------------------------------------------------",
            "HEADER_CONFIG": " CONFIGURATION REPORT",
            "LBL_ITERATIONS": " Iterations: ",
            "LBL_GRAPH_VIS": " Graph Viz:  ",
            "LBL_RECURSION": " Rec. Limit: ",
            "HEADER_PUZZLES": " Puzzles to Test:",
            "FMT_PUZZLE_ITEM": "   - {name} ({holes} holes)",
            "HEADER_MODES": " Modes to Test:",
            "FMT_MODE_ITEM": "   - {name:<10} [MVC: {mvc:<5} | Lookahead: {look:<5}]",
            "MSG_TEST_INIT": "\n   >>> INITIALIZING TEST: {name}",
            "MSG_MODE_RUN": "       > Running Mode: {name}...",
            "MSG_ITER_SETUP": "         [Iter {current}/{total}]",
            "MSG_RENDERING": "         [Graph] Rendering decision tree...",
            "MSG_GRAPH_ERROR": "         [Error] Graphviz failed: {error}",
            "HEADER_ANALYSIS": " ANALYSIS: {name}",
            "TBL_HDR_METRIC": "METRIC",
            "FMT_TBL_COL_NAME": " {val:^15} |",
            "TBL_ROW_TIME": "Time (sec)",
            "FMT_TBL_VAL_FLOAT": " {val:^15.5f} |",
            "TBL_ROW_BACKTRACKS": "Backtracks",
            "FMT_TBL_VAL_INT": " {val:^15} |",
            "TBL_ROW_NODES": "Nodes Visited",
            "TBL_ROW_SKIPPED": "Nodes Pruned",
            "TBL_ROW_ACCURACY": "Accuracy",
            "FMT_TBL_VAL_PCT": " {val:^14.1f}% |",
            "HEADER_COMPARISON": " COMPARATIVE GAINS (vs Base)",
            "FMT_COMPARISON_ITEM": "   > {name:<10}: {diff} fewer backtracks ({pct:.1f}% reduction)",
            "HEADER_FINAL": " FINAL SUMMARY TABLE "
        }
    },

    "output_settings": {
        "directory": "test_artifacts",
        "file_format": "txt",
        "batch_iterations": true,
        "filename_config": {
            "include_testset_name": true,
            "include_modifiers": true,
            "include_date": true,
            "include_iteration_index": false
        }
    },

    "puzzles_to_generate": [
        {"name": "Easy", "holes": 30},
        {"name": "Medium", "holes": 45},
        {"name": "Hard", "holes": 55}
    ],
    "modes_to_test": [
        {"name": "Base", "mvc": false, "look": false},
        {"name": "MVC", "mvc": true, "look": false},
        {"name": "MVC+Look", "mvc": true, "look": true}
    ]
}

5. The Test Harness (test_harness.py)
Description: This is the "Controller" or Main Loop. It reads the config, calls the generator to make puzzles, loops through the test modes, invokes the solver, measures performance, and calls the Visualizer to save the data.


```
# This is formatted as code
```
FUNCTION run_suite():
    Load Configuration from 'config.json'
    
    PRINT "Generating Puzzles..."
    List_of_Puzzles = []
    FOR each puzzle_setting in config:
        Data = CALL generator.generate(puzzle_setting.holes)
        ADD (Name, Data) to List_of_Puzzles

    PRINT "Starting Tests..."
    
    FOR each Puzzle in List_of_Puzzles:
        
        FOR each Mode in config['modes_to_test']:
            
            Initialize Batch_Data_List
            
            FOR i from 1 to config['iterations']:
                
                Initialize Solver with Puzzle Data
                
                (Visualization - First Iteration Only)
                IF i == 1 AND config['visualize_graph']:
                    CALL Visualizer.display_board("Unfinished")

                START Timer
                CALL Solver.solve(Mode.mvc, Mode.lookahead)
                STOP Timer
                
                (Visualization - First Iteration Only)
                IF i == 1 AND config['visualize_graph']:
                    CALL Visualizer.display_board("Solved")
                    CALL Visualizer.render_graph(Solver.metrics)

                (Data Collection)
                Create Result_Object {Time, Backtracks, Grid, Mode_Info}
                ADD Result_Object to Batch_Data_List

                IF Batching is OFF:
                    Generate Filename
                    CALL Visualizer.save_batch([Result_Object])

            (End of Iteration Loop)
            
            IF Batching is ON:
                Generate Filename (based on Date, Mode, PuzzleName)
                CALL Visualizer.save_batch(Batch_Data_List)

            Calculate Average Time
            ADD Summary Stats to Final_Results_Table

    DISPLAY Final_Results_Table (Pandas DataFrame)


In [ ]:
%%writefile puzzles.txt
puzzle:003020600900305001001806400008102900700000008006708200002609500800203009005010300, solution:483921657967345821251876493548132976729564138136798245372689514814253769695417382
puzzle:200080300060070084030500209000105408000000000402706000301007040720040060004010003, solution:297481356165972984834569217973125468516834792482796531351687942729345168648219753
puzzle:000000907000420180000705026100904000050000040000507009920108000034059000507000000, solution:642381957795426183381795426168934275259671843374582619926148357834259761517863924
puzzle:030050040008010500460000012070502080000603000050109030920000065005080200010090070, solution:231756849798214536465839712174562983829673451653149728927341865345987216516498372
puzzle:000000080800701040040020030374000900000030000005000321010060050050802006080000000, solution:721345689839761245645928137374652918296138475158479326413269758957812463582594167

In [ ]:
%%writefile loader.py
import pandas as pd
import os

class PuzzleLoader:
    def __init__(self, config):
        self.config = config
        self.settings = config["INPUT_SETTINGS"]
        self.format_specs = self.settings["FORMAT_SPECS"]

    def load(self):
        """Loads puzzles based on configuration settings."""
        if not self.settings.get("ENABLE_FILE_INPUT", False):
            return None

        file_path = self.settings.get("FILE_PATH", "puzzles.txt")

        if not os.path.exists(file_path):
            print(f"Error: Puzzle file not found at {file_path}")
            return []

        if file_path.endswith('.csv'):
            return self._load_csv(file_path)
        elif file_path.endswith('.xlsx') or file_path.endswith('.xls'):
            return self._load_excel(file_path)
        else:
            return self._load_txt(file_path)

    def _load_csv(self, path):
        df = pd.read_csv(path)
        return self._extract_from_df(df, "CSV_EXCEL")

    def _load_excel(self, path):
        df = pd.read_excel(path)
        return self._extract_from_df(df, "CSV_EXCEL")

    def _extract_from_df(self, df, spec_key):
        specs = self.format_specs[spec_key]
        p_col = specs["PUZZLE_COL"]
        s_col = specs["SOLUTION_COL"]

        puzzles = []
        for _, row in df.iterrows():
            entry = {'puzzle': str(row[p_col])}
            if s_col in df.columns and pd.notna(row[s_col]):
                entry['solution'] = str(row[s_col])
            puzzles.append(entry)
        return puzzles

    def _load_txt(self, path):
        specs = self.format_specs["TXT"]
        p_prefix = specs["PUZZLE_PREFIX"]
        s_prefix = specs["SOLUTION_PREFIX"]
        delimiter = specs["DELIMITER"]

        puzzles = []
        with open(path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line: continue

                parts = line.split(delimiter)
                entry = {}

                for part in parts:
                    part = part.strip()
                    if part.startswith(p_prefix):
                        entry['puzzle'] = part[len(p_prefix):].strip()
                    elif part.startswith(s_prefix):
                        entry['solution'] = part[len(s_prefix):].strip()

                if 'puzzle' in entry:
                    puzzles.append(entry)

        return puzzles

In [ ]:
%%writefile utils.py
import json

def load_config(path='config.json'):
    """Loads the test configuration from a JSON file."""
    try:
        with open(path, 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"Error: Configuration file '{path}' not found.")
        return {}

In [ ]:
%%writefile analysis_reporter.py
import pandas as pd
from IPython.display import display

def get_templates(config):
    """Helper to extract text templates from config."""
    return config.get('VISUAL_CONSTANTS', {}).get('TEXT_TEMPLATES', {})

def print_config_header(config):
    """Prints the full configuration using templates from config."""
    c = config.get('COLORS', {})
    txt = get_templates(config)
    bold = c.get('BOLD', '')
    reset = c.get('RESET', '')
    blue = c.get('BLUE', '')

    # Header Block
    print(f"\n{bold}{blue}{txt.get('SEP_EQUALS', '')}{reset}")
    print(f"{bold}{blue}{txt.get('HEADER_CONFIG', 'CONFIG')}{reset}")
    print(f"{bold}{blue}{txt.get('SEP_EQUALS', '')}{reset}")

    # Global Settings
    print(f"{bold}{txt.get('LBL_ITERATIONS', 'Iterations:')}{reset} {config.get('iterations', 0)}")
    print(f"{bold}{txt.get('LBL_GRAPH_VIS', 'Graph:')}{reset} {config.get('visualize_graph', False)}")
    print(f"{bold}{txt.get('LBL_RECURSION', 'Recursion:')}{reset}     {config.get('METRICS_CONSTANTS', {}).get('RECURSION_LIMIT', 1000)}")

    # Puzzles
    print(f"{bold}{txt.get('HEADER_PUZZLES', 'Puzzles:')}{reset}")
    for p in config.get('puzzles_to_generate', []):
        print(txt.get('FMT_PUZZLE_ITEM', '').format(name=p['name'], holes=p['holes']))

    # Modes
    print(f"{bold}{txt.get('HEADER_MODES', 'Modes:')}{reset}")
    for m in config.get('modes_to_test', []):
        print(txt.get('FMT_MODE_ITEM', '').format(name=m['name'], mvc=str(m['mvc']), look=str(m['look'])))

    print(f"{bold}{blue}{txt.get('SEP_EQUALS', '')}{reset}\n")

def print_puzzle_init(puzzle_name, config):
    txt = get_templates(config)
    print(txt.get('MSG_TEST_INIT', '').format(name=puzzle_name))

def print_mode_start(mode_name, config):
    txt = get_templates(config)
    print(txt.get('MSG_MODE_RUN', '').format(name=mode_name))

def print_iteration_setup(current, total, config):
    txt = get_templates(config)
    print(txt.get('MSG_ITER_SETUP', '').format(current=current, total=total))

def print_graph_rendering_msg(config):
    txt = get_templates(config)
    print(txt.get('MSG_RENDERING', 'Rendering...'))

def print_graph_error(error, config):
    txt = get_templates(config)
    print(txt.get('MSG_GRAPH_ERROR', '').format(error=error))

def generate_analysis_report(puzzle_name, all_mode_results, config):
    """
    Generates a comparative analysis of the puzzle execution.
    """
    c = config.get('COLORS', {})
    txt = get_templates(config)
    bold = c.get('BOLD', '')
    reset = c.get('RESET', '')
    yellow = c.get('YELLOW', '')

    # Analysis Header
    print(f"{bold}{yellow}{txt.get('HEADER_ANALYSIS', '').format(name=puzzle_name)}{reset}")
    print(txt.get('SEP_DASHES', '-'))

    # 1. Table Header
    print(f"{txt.get('TBL_HDR_METRIC', 'METRIC'):<25} |", end="")
    for mode_name in all_mode_results.keys():
        print(txt.get('FMT_TBL_COL_NAME', '').format(val=mode_name[:12]), end="")
    print(f"\n{txt.get('SEP_DASHES', '-')}")

    # 2. Avg Time
    print(f"{txt.get('TBL_ROW_TIME', 'Time'):<25} |", end="")
    for mode in all_mode_results.values():
        print(txt.get('FMT_TBL_VAL_FLOAT', '').format(val=mode['avg_time']), end="")
    print()

    # 3. Backtracks
    print(f"{txt.get('TBL_ROW_BACKTRACKS', 'Backtracks'):<25} |", end="")
    for mode in all_mode_results.values():
        print(txt.get('FMT_TBL_VAL_INT', '').format(val=int(mode['avg_backtracks'])), end="")
    print()

    # 4. Nodes Visited (Tree Size)
    print(f"{txt.get('TBL_ROW_NODES', 'Nodes'):<25} |", end="")
    for mode in all_mode_results.values():
        nodes = mode['metrics'].get('nodes_visited', 0)
        print(txt.get('FMT_TBL_VAL_INT', '').format(val=nodes), end="")
    print()

    # 5. Nodes Skipped (Pruned)
    print(f"{txt.get('TBL_ROW_SKIPPED', 'Skipped'):<25} |", end="")
    base_nodes = list(all_mode_results.values())[0]['metrics'].get('nodes_visited', 0)
    for mode in all_mode_results.values():
        current_nodes = mode['metrics'].get('nodes_visited', 0)
        skipped = max(0, base_nodes - current_nodes)
        print(txt.get('FMT_TBL_VAL_INT', '').format(val=skipped), end="")
    print()

    # 6. Accuracy (Success Rate)
    print(f"{txt.get('TBL_ROW_ACCURACY', 'Accuracy'):<25} |", end="")
    for mode in all_mode_results.values():
        acc = mode['accuracy'] * 100
        print(txt.get('FMT_TBL_VAL_PCT', '').format(val=acc), end="")
    print(f"\n{txt.get('SEP_DASHES', '-')}")

    # 7. Deltas / Comparisons
    print(f"{bold}{txt.get('HEADER_COMPARISON', 'COMPARISONS')}{reset}")
    modes_list = list(all_mode_results.keys())
    if modes_list:
        base_mode = modes_list[0]
        base_bt = all_mode_results[base_mode]['avg_backtracks']

        for mode_name in modes_list[1:]:
            curr_bt = all_mode_results[mode_name]['avg_backtracks']
            diff = base_bt - curr_bt
            reduction_pct = (diff / base_bt * 100) if base_bt > 0 else 0
            print(txt.get('FMT_COMPARISON_ITEM', '').format(name=mode_name, diff=int(diff), pct=reduction_pct))

    print(f"{txt.get('SEP_DASHES', '-')}\n")

def print_final_summary(results_summary, config):
    txt = get_templates(config)
    print("\n" + txt.get('SEP_EQUALS', '='))
    print(txt.get('HEADER_FINAL', 'FINAL'))
    print(txt.get('SEP_EQUALS', '='))
    display(pd.DataFrame(results_summary))

In [ ]:
%%writefile test_harness.py
import time
import datetime
from statistics import mean

# --- LOCAL IMPORTS ---
from utils import load_config
import analysis_reporter as reporter
from solver import SudokuSolver
from visualizer import OutputManager
import generator

def run_suite():
    # 1. Load Config
    config = load_config()
    txt = config['VISUAL_CONSTANTS']['TEXT_TEMPLATES']
    batch_mode = config['output_settings']['batch_iterations']
    show_graph = config.get('visualize_graph', False)
    use_color = config.get('use_color_output', True)

    # New Toggle for showing generated puzzles
    show_gen = config['VISUAL_CONSTANTS'].get('show_generated_puzzles', False)
    gen_char = config['VISUAL_CONSTANTS'].get('GENERATOR_CHAR', '.')

    # 2. Print Header
    reporter.print_config_header(config)

    # 3. Generate Puzzles
    print(txt['MSG_GENERATING'])
    puzzles = []

    for p in config['puzzles_to_generate']:
        # Unpack the tuple (unsolved, solved) from generator
        unsolved_str, solved_str = generator.generate_test_puzzle(p['holes'], config)

        # Store only the unsolved part for the actual test loop
        puzzles.append((p['name'], unsolved_str))

        if show_gen:
            print(f"\n   [{p['name']}]")
            # Replace 0 with the custom char for display
            display_unsolved = unsolved_str.replace('0', gen_char)
            print(f"   Unsolved: {display_unsolved}")
            print(f"   Solved:   {solved_str}")

    print("\n" + txt['SEP_DASHES'])

    results_summary = []
    suite_timestamp = datetime.datetime.now().strftime("%Y%m%d%H%M%S")

    # 4. Main Test Loop
    for p_name, p_data in puzzles:
        reporter.print_puzzle_init(p_name, config)

        puzzle_stats = {}

        # Loop Modes
        for mode in config['modes_to_test']:
            reporter.print_mode_start(mode['name'], config)

            times = []
            backtracks = []
            batch_data = []
            success_count = 0
            last_metrics = {}

            # Loop Iterations
            for i in range(config['iterations']):
                solver = SudokuSolver(p_data)

                # --- VISUALS: Unfinished Board (First Iteration Only) ---
                if i == 0 and show_graph:
                    reporter.print_iteration_setup(i+1, config['iterations'], config)
                    OutputManager.display_board(solver, config, f"UNFINISHED ({p_name} - {mode['name']})", use_color)

                # --- EXECUTION ---
                start = time.perf_counter()
                success = solver.solve(use_mvc=mode['mvc'], use_lookahead=mode['look'])
                end = time.perf_counter()

                # --- METRICS ---
                duration = end - start
                times.append(duration)
                backtracks.append(solver.metrics['backtracks'])
                if success: success_count += 1
                last_metrics = solver.metrics

                # --- VISUALS: Solved Board & Graph (First Iteration Only) ---
                if i == 0 and show_graph:
                    OutputManager.display_board(solver, config, f"SOLVED ({p_name} - {mode['name']})", use_color)
                    try:
                        reporter.print_graph_rendering_msg(config)
                        # FIX: Passed 'config' as the second argument here to fix the graph error
                        OutputManager.render_graph(solver.metrics, config)
                    except Exception as e:
                        reporter.print_graph_error(e, config)

                # --- COLLECT RAW DATA ---
                result_entry = {
                    "iteration": i + 1,
                    "puzzle": p_name,
                    "mode_name": mode['name'],
                    "mvc": mode['mvc'],
                    "look": mode['look'],
                    "time": duration,
                    "backtracks": solver.metrics['backtracks'],
                    "grid_str": "\n".join(["".join(map(str, row)) for row in solver.grid])
                }

                if batch_mode:
                    batch_data.append(result_entry)
                else:
                    fname = OutputManager.generate_filename(config, p_name, mode, suite_timestamp, iteration_idx=i+1)
                    OutputManager.save_batch(config, [result_entry], fname)

            # --- SAVE BATCH ---
            if batch_mode:
                fname = OutputManager.generate_filename(config, p_name, mode, suite_timestamp)
                OutputManager.save_batch(config, batch_data, fname)

            # --- AGGREGATE STATS ---
            avg_time = mean(times)
            avg_bt = mean(backtracks)

            puzzle_stats[mode['name']] = {
                "avg_time": avg_time,
                "avg_backtracks": avg_bt,
                "accuracy": success_count / config['iterations'],
                "metrics": last_metrics
            }

            results_summary.append({
                "Puzzle": p_name,
                "Mode": mode["name"],
                "Avg Time": avg_time,
                "Backtracks": avg_bt
            })

        # 5. Print Analysis for this Puzzle
        reporter.generate_analysis_report(p_name, puzzle_stats, config)

    # 6. Final Global Summary
    reporter.print_final_summary(results_summary, config)

if __name__ == "__main__":
    run_suite()

In [ ]:
%run test_harness.py